### Grading
The final score that you will receive for your programming assignment is generated in relation to the total points set in your programming assignment item—not the total point value in the nbgrader notebook.<br>
When calculating the final score shown to learners, the programming assignment takes the percentage of earned points vs. the total points provided by nbgrader and returns a score matching the equivalent percentage of the point value for the programming assignment. <br>
**DO NOT CHANGE VARIABLE OR METHOD SIGNATURES** The autograder will not work properly if your change the variable or method signatures. 

### Validate Button
Please note that this assignment uses nbgrader to facilitate grading. You will see a **validate button** at the top of your Jupyter notebook. If you hit this button, it will run tests cases for the lab that aren't hidden. It is good to use the validate button before submitting the lab. Do know that the labs in the course contain hidden test cases. The validate button will not let you know whether these test cases pass. After submitting your lab, you can see more information about these hidden test cases in the Grader Output. <br>
***Cells with longer execution times will cause the validate button to time out and freeze. Please know that if you run into Validate time-outs, it will not affect the final submission grading.*** <br>

# Building Recommender Systems for Movie Rating Prediction

In this assignment, we will build a recommender systems that predict movie ratings. [MovieLense](https://grouplens.org/datasets/movielens/) has currently 25 million user-movie ratings.  Since the entire data is too big, we use  a 1 million ratings subset [MovieLens 1M](https://www.kaggle.com/odedgolden/movielens-1m-dataset), and we reformatted the data to make it more convenient to use.

In [2]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import time
from sklearn.model_selection import train_test_split
from scipy.sparse import coo_matrix, csr_matrix
from scipy.spatial.distance import jaccard, cosine 
from pytest import approx

In [3]:
MV_users = pd.read_csv('data/users.csv')
MV_movies = pd.read_csv('data/movies.csv')
train = pd.read_csv('data/train.csv')
test = pd.read_csv('data/test.csv')

In [4]:
from collections import namedtuple
Data = namedtuple('Data', ['users','movies','train','test'])
data = Data(MV_users, MV_movies, train, test)

In [5]:
print(data.train.describe())
MV_movies

                 uID            mID         rating
count  700146.000000  700146.000000  700146.000000
mean     3022.960334    1865.307324       3.581589
std      1729.128758    1096.507590       1.117508
min         1.000000       1.000000       1.000000
25%      1503.000000    1029.000000       3.000000
50%      3067.000000    1834.000000       4.000000
75%      4474.000000    2770.000000       4.000000
max      6040.000000    3952.000000       5.000000


,mID,title,year,Doc,Com,Hor,Adv,Wes,Dra,Ani,...,Chi,Cri,Thr,Sci,Mys,Rom,Fil,Fan,Act,Mus
0,1,Toy Story,1995,0,1,0,0,0,0,1,...,1,0,0,0,0,0,0,0,0,0
1,2,Jumanji,1995,0,0,0,1,0,0,0,...,1,0,0,0,0,0,0,1,0,0
2,3,Grumpier Old Men,1995,0,1,0,0,0,0,0,...,0,0,0,0,0,1,0,0,0,0
3,4,Waiting to Exhale,1995,0,1,0,0,0,1,0,...,0,0,0,0,0,0,0,0,0,0
4,5,Father of the Bride Part II,1995,0,1,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3878,3948,Meet the Parents,2000,0,1,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
3879,3949,Requiem for a Dream,2000,0,0,0,0,0,1,0,...,0,0,0,0,0,0,0,0,0,0
3880,3950,Tigerland,2000,0,0,0,0,0,1,0,...,0,0,0,0,0,0,0,0,0,0
3881,3951,Two Family House,2000,0,0,0,0,0,1,0,...,0,0,0,0,0,0,0,0,0,0


### Starter codes
Now, we will be building a recommender system which has various techniques to predict ratings. 
The `class RecSys` has baseline prediction methods (such as predicting everything to 3 or to average rating of each user) and other utility functions. `class ContentBased` and `class Collaborative` inherit `class RecSys` and further add methods calculating item-item similarity matrix. You will be completing those functions using what we learned about content-based filtering and collaborative filtering.

`RecSys`'s `rating_matrix` method converts the (user id, movie id, rating) triplet from the train data (train data's ratings are known) into a utility matrix for 6040 users and 3883 movies.    
Here, we create the utility matrix as a dense matrix (numpy.array) format for convenience. But in a real world data where hundreds of millions of users and items may exist, we won't be able to create the utility matrix in a dense matrix format (For those who are curious why, try measuring the dense matrix self.Mr using .nbytes()). In that case, we may use sparse matrix operations as much as possible and distributed file systems and distributed computing will be needed. Fortunately, our data is small enough to fit in a laptop/pc memory. Also, we will use numpy and scipy.sparse, which allow significantly faster calculations than calculating on pandas.DataFrame object.    
In the `rating_matrix` method, pay attention to the index mapping as user IDs and movie IDs are not the same as array index.

In [6]:
class RecSys():
    def __init__(self,data):
        self.data=data
        self.allusers = list(self.data.users['uID'])
        self.allmovies = list(self.data.movies['mID'])
        self.genres = list(self.data.movies.columns.drop(['mID', 'title', 'year']))
##?? Why can't we just zip self.allmovies, self.allusers ??
        self.mid2idx = dict(zip(self.data.movies.mID,list(range(len(self.data.movies)))))
        self.uid2idx = dict(zip(self.data.users.uID,list(range(len(self.data.users)))))
        self.Mr=self.rating_matrix()
##?? Why is this initialized in the RecSys class if it is only used in the ContentBased class??
        self.Mm=None 
        self.sim=np.zeros((len(self.allmovies),len(self.allmovies)))
        
    def rating_matrix(self):
        """
        Convert the rating matrix to numpy array of shape (#allusers,#allmovies)
        """
        ind_movie = [self.mid2idx[x] for x in self.data.train.mID] 
        ind_user = [self.uid2idx[x] for x in self.data.train.uID]
        rating_train = list(self.data.train.rating)
        
        return np.array(coo_matrix((rating_train, (ind_user, ind_movie)), shape=(len(self.allusers), len(self.allmovies))).toarray())


    def predict_everything_to_3(self):
        """
        Predict everything to 3 for the test data
        """
        # Generate an array with 3s against all entries in test dataset
        # your code here
        yp = [3 for x in range(len(self.data.test.rating))]
        return np.array(yp)
        
    def predict_to_user_average(self):
        """
        Predict to average rating for the user.
        Returns numpy array of shape (#users,)
        """
        # Generate an array as follows:
        # 1. Calculate all avg user rating as sum of ratings of user across all movies/number of movies whose rating > 0
        # 2. Return the average rating of users in test data
        # your code here
        uIDintest = list(pd.unique(self.data.test.uID))
        uIDavg = [self.data.train.rating[self.data.train.uID == uIDs].mean() for uIDs in uIDintest]
        user_average = dict(zip(uIDintest, uIDavg))
        yp = [user_average[x] for x in self.data.test.uID]
        return np.array(yp)
    
    def predict_from_sim(self,uid,mid):
        """
        Predict a user rating on a movie given userID and movieID
        """
        # Predict user rating as follows:
        # 1. Get entry of user id in rating matrix
        # 2. Get entry of movie id in sim matrix
        # 3. Employ 1 and 2 to predict user rating of the movie
        # your code here
 ##?? The Collaborative classs is supposed to use this predict_from_self method, which is working correctly for the ContentBased class
 ##?? Could we fix this so it works for both classes?  (The solution will probably involve changing the cosine similarity matrix.)       
        uidx = self.uid2idx[uid]
        midx = self.mid2idx[mid]

        # Include the case where the user has already rated that movie, return that rating
        if self.Mr[uidx][midx] != 0:
            return self.Mr[uidx][midx]
    
        # row all movie rating of uidx
        user_ratings = self.Mr[uidx]
        
        # edge case of no ratings
        if np.sum(user_ratings) == 0:
            return 3
        
        # row of similarity matrix for the movie we want to estimate the rating of
        sim_movies = self.sim[midx]

        # weighted inner product of the ratings and the similarities 
        rated_movie_index = np.where(user_ratings != 0)[0]
        raw = 0
        weight = 0
        for k in rated_movie_index:
            raw = raw + user_ratings[k]*sim_movies[k]
            weight = weight + sim_movies[k]

        
        # What about the case where nothing in similar
        if weight == 0:
            return 3        
        return raw/weight

    
    def predict(self):
        """
        Predict ratings in the test data. Returns predicted rating in a numpy array of size (# of rows in testdata,)
        """
        y_pred = [0] * len(self.data.test.uID)
        for i in range(len(self.data.test.uID)):
            y_pred[i] = self.predict_from_sim(self.data.test.uID[i], self.data.test.mID[i])

        return np.array(y_pred)    

     
    
    def rmse(self,yp):
        yp[np.isnan(yp)]=3 #In case there is nan values in prediction, it will impute to 3.
        yt=np.array(self.data.test.rating)
        return np.sqrt(((yt-yp)**2).mean())

    
class ContentBased(RecSys):
    def __init__(self,data):
        super().__init__(data)
        self.data=data
        self.Mm = self.calc_movie_feature_matrix()  
        
    def calc_movie_feature_matrix(self):
        """
        Create movie feature matrix in a numpy array of shape (#allmovies, #genres) 
        """
        return self.data.movies.drop(columns = ['mID', 'title', 'year'])
    
    def calc_item_item_similarity(self):
        """
        Create item-item similarity using Jaccard similarity
        """
        # Update the sim matrix by calculating item-item similarity using Jaccard similarity
        # Jaccard Similarity: J(A, B) = |A∩B| / |A∪B| 
        # your code here
#        # Version 2
#        for i in range(len(self.allmovies)):
#            for j in range(i,len(self.allmovies)):
#                self.sim[i][j] = jaccard(self.Mm.loc[i], self.Mm.loc[j])
#                self.sim[j][i] = self.sim[i][j]

#        # This is way too slow!
#        for i in range(len(self.allmovies)):
#            for j in range(len(self.allmovies)):
#                intersection = sum([self.Mm.loc[i][k]*self.Mm.loc[j][k] for k in self.genres])
#                union = sum( [max (self.Mm.loc[i][k],self.Mm.loc[j][k]) for k in self.genres])
#                self.sim[i][j] = intersection/union
#        # Note that I am doing twice as many calculations as I need to 
        from sklearn.metrics import pairwise_distances
        Mm_array = np.array(self.Mm)
        self.sim = 1-pairwise_distances(Mm_array, metric = 'jaccard')
                
class Collaborative(RecSys):    
    def __init__(self,data):
        super().__init__(data)
##?? When we defined class ContentBased, we also had self.data = data.  Was that redundant??       
    def calc_item_item_similarity(self, simfunction, *X):  
        """
        Create item-item similarity using similarity function. 
        X is an optional transformed matrix of Mr
        """    
        # General function that calculates item-item similarity based on the sim function and data inputed
        if len(X)==0:
            self.sim = simfunction()            
        else:
            self.sim = simfunction(X[0]) # *X passes in a tuple format of (X,), to X[0] will be the actual transformed matrix
            
    def cossim(self):    
        """
        Calculates item-item similarity for all pairs of items using cosine similarity (values from 0 to 1) on utility matrix
        Returns a cosine similarity matrix of size (#all movies, #all movies)
        """
        # Return a sim matrix by calculating item-item similarity for all pairs of items using Jaccard similarity
        # Cosine Similarity: C(A, B) = (A.B) / (||A||.||B||) 
        # your code here
        X = self.Mr.copy()
        masked = np.ma.masked_equal(X,0)
        row_means = masked.mean(axis = 1).data
        for i in range(len(self.allusers)):
            X[i][np.where(X[i] != 0)] = X[i][np.where(X[i] != 0)] - row_means[i]
        
        # X = X/np.sqrt((X**2).sum(axis=0))
        # X[np.isnan(X)] = 0

        from sklearn.metrics.pairwise import cosine_similarity
        cos_sim = cosine_similarity(X.T,X.T)
        print(cos_sim.shape)
        print(np.trace(cos_sim))

        for i in range(len(self.allmovies)):
            cos_sim[i][i] = 1
        print(np.trace(cos_sim))
        return np.array(cos_sim)
    
    def jacsim(self,Xr):
        """
        Calculates item-item similarity for all pairs of items using jaccard similarity (values from 0 to 1)
        Xr is the transformed rating matrix.
        """    
        # Return a sim matrix by calculating item-item similarity for all pairs of items using Jaccard similarity
        # Jaccard Similarity: J(A, B) = |A∩B| / |A∪B| 
        # your code here
        
        from sklearn.metrics import pairwise_distances
        Xr = np.array(Xr)
        jaccard_sim = 1-pairwise_distances(Xr, metric = 'jaccard')
        return jaccard_sim
 
 

In [7]:
RS = RecSys(data)


In [8]:
sample_array = np.array([[0,4,2,0,0,0],[3,0,0,0,0,5],[1,2,3,0,0,0]])
sample_array.sum(axis = 1)
masked = np.ma.masked_equal(sample_array,0)
A = masked.mean(axis = 1).data 
print(A[1])
sample_array[1][np.where(sample_array[1] == 0)] = 16
sample_array[1]


4.0


array([ 3, 16, 16, 16, 16,  5])

In [9]:
sample_array = np.array([[0,4,2,0,0,0],[3,0,0,0,0,5],[1,2,3,0,0,0]])
masked = np.ma.masked_equal(sample_array,0)
A = masked.mean(axis = 1).data
for i in range(3):
    sample_array[i][np.where(sample_array[i] != 0)] = sample_array[i][np.where(sample_array[i] != 0)] - A[i]
print(sample_array)

from sklearn.metrics.pairwise import cosine_distances
c_sim = 1-cosine_distances(sample_array, sample_array)
print(c_sim)

B = coo_matrix(([1,-1,1,1,-1,-1,2,-2,1,-1],([1,1,3,3,3,3,4,4,5,5],[1,2,1,2,5,4,3,4,4,5])))
B = B.toarray()
print(B.T)
B_c_sim = 1-cosine_distances(B,B)
print(B_c_sim)


[[ 0  1 -1  0  0  0]
 [-1  0  0  0  0  1]
 [-1  0  1  0  0  0]]
[[ 1.   0.  -0.5]
 [ 0.   1.   0.5]
 [-0.5  0.5  1. ]]
[[ 0  0  0  0  0  0]
 [ 0  1  0  1  0  0]
 [ 0 -1  0  1  0  0]
 [ 0  0  0  0  2  0]
 [ 0  0  0 -1 -2  1]
 [ 0  0  0 -1  0 -1]]
[[ 1.          0.          0.          0.          0.          0.        ]
 [ 0.          1.          0.          0.          0.          0.        ]
 [ 0.          0.          1.          0.          0.          0.        ]
 [ 0.          0.          0.          1.          0.35355339  0.        ]
 [ 0.          0.          0.          0.35355339  1.         -0.5       ]
 [ 0.          0.          0.          0.         -0.5         1.        ]]


In [10]:
X = B.copy()
X = X/np.sqrt((X**2).sum(axis=0))
print(X)
X[np.isnan(X)] = 0
print(X) 
1-cosine_distances(X.T,X.T)

[[        nan  0.          0.          0.          0.          0.        ]
 [        nan  0.70710678 -0.70710678  0.          0.          0.        ]
 [        nan  0.          0.          0.          0.          0.        ]
 [        nan  0.70710678  0.70710678  0.         -0.40824829 -0.70710678]
 [        nan  0.          0.          1.         -0.81649658  0.        ]
 [        nan  0.          0.          0.          0.40824829 -0.70710678]]
[[ 0.          0.          0.          0.          0.          0.        ]
 [ 0.          0.70710678 -0.70710678  0.          0.          0.        ]
 [ 0.          0.          0.          0.          0.          0.        ]
 [ 0.          0.70710678  0.70710678  0.         -0.40824829 -0.70710678]
 [ 0.          0.          0.          1.         -0.81649658  0.        ]
 [ 0.          0.          0.          0.          0.40824829 -0.70710678]]


C:\Users\eltur\AppData\Local\Temp\ipykernel_21148\3227350230.py:2: RuntimeWarning: invalid value encountered in divide
  X = X/np.sqrt((X**2).sum(axis=0))


array([[ 0.        ,  0.        ,  0.        ,  0.        ,  0.        ,
         0.        ],
       [ 0.        ,  1.        ,  0.        ,  0.        , -0.28867513,
        -0.5       ],
       [ 0.        ,  0.        ,  1.        ,  0.        , -0.28867513,
        -0.5       ],
       [ 0.        ,  0.        ,  0.        ,  1.        , -0.81649658,
         0.        ],
       [ 0.        , -0.28867513, -0.28867513, -0.81649658,  1.        ,
         0.        ],
       [ 0.        , -0.5       , -0.5       ,  0.        ,  0.        ,
         1.        ]])

(3,4,5) * (0,1,0.5) = 0+4+2.5 = 6.5
6.5/3 = 2.167
6.5/1.5 = 4.333

(3,4,5) * (1,0,0.5) = 3 + 0 + 2.5 = 5.5
5.5/1.5 = 3.67

In [11]:
x = np.array([1,2,3,4])
np.where(x>2)[0]

array([2, 3])

# Q1. Baseline models [15 pts]

### 1a. Complete the function `predict_everything_to_3` in the class `RecSys`  [5 pts]

In [12]:
# Creating Sample test data
np.random.seed(42)
sample_train = train[:30000]
sample_test = test[:30000]


sample_MV_users = MV_users[(MV_users.uID.isin(sample_train.uID)) | (MV_users.uID.isin(sample_test.uID))]
sample_MV_movies = MV_movies[(MV_movies.mID.isin(sample_train.mID)) | (MV_movies.mID.isin(sample_test.mID))]


sample_data = Data(sample_MV_users, sample_MV_movies, sample_train, sample_test)

In [13]:
# Sample tests predict_everything_to_3 in class RecSys

sample_rs = RecSys(sample_data)
sample_yp = sample_rs.predict_everything_to_3()
print(sample_rs.rmse(sample_yp))
assert sample_rs.rmse(sample_yp)==approx(1.2642784503423288, abs=1e-3), "Did you predict everything to 3 for the test data?"

1.2642784503423288


In [14]:
# Hidden tests predict_everything_to_3 in class RecSys
rs = RecSys(data)
yp = rs.predict_everything_to_3()
print(rs.rmse(yp))

1.2585510334053043


### 1b. Complete the function predict_to_user_average in the class RecSys [10 pts]
Hint: Include rated items only when averaging

In [15]:
# Sample tests predict_to_user_average in the class RecSys
sample_yp = sample_rs.predict_to_user_average()
print(sample_rs.rmse(sample_yp))
assert sample_rs.rmse(sample_yp)==approx(1.1429596846619763, abs=1e-3), "Check predict_to_user_average in the RecSys class. Did you predict to average rating for the user?" 

1.1429596846619763


In [16]:
# Hidden tests predict_to_user_average in the class RecSys
yp = rs.predict_to_user_average()
print(rs.rmse(yp))

1.0352910334228647


# Q2. Content-Based model [25 pts]

### 2a. Complete the function calc_movie_feature_matrix in the class ContentBased [5 pts]

In [17]:
cb = ContentBased(data)

In [18]:
from sklearn.metrics import pairwise_distances
Hh = np.array(cb.Mm)

check = 1-pairwise_distances(Hh, metric = 'jaccard')
check?

c:\Users\eltur\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\metrics\pairwise.py:2466: DataConversionWarning: Data was converted to boolean for metric jaccard
  warnings.warn(msg, DataConversionWarning)


Type:        ndarray
String form:
[[1.   0.2  0.25 ... 0.   0.   0.  ]
           [0.2  1.   0.   ... 0.   0.   0.  ]
           [0.25 0.   1.   ... 0. <...>    ... 1.   1.   0.5 ]
           [0.   0.   0.   ... 1.   1.   0.5 ]
           [0.   0.   0.   ... 0.5  0.5  1.  ]]
Length:      3883
File:        c:\users\eltur\appdata\local\programs\python\python312\lib\site-packages\numpy\__init__.py
Docstring:  
ndarray(shape, dtype=float, buffer=None, offset=0,
        strides=None, order=None)

An array object represents a multidimensional, homogeneous array
of fixed-size items.  An associated data-type object describes the
format of each element in the array (its byte-order, how many bytes it
occupies in memory, whether it is an integer, a floating point number,
or something else, etc.)

Arrays should be constructed using `array`, `zeros` or `empty` (refer
to the See Also section below).  The parameters given here refer to
a low-level method (`ndarray(...)`) for instantiating an array.



In [19]:
# tests calc_movie_feature_matrix in the class ContentBased 
assert(cb.Mm.shape==(3883, 18))

### 2b. Complete the function calc_item_item_similarity in the class ContentBased [10 pts]
This function updates `self.sim` and does not return a value.    
Some factors to think about:     
1. The movie feature matrix has binary elements. Which similarity metric should be used?
2. What is the computation complexity (time complexity) on similarity calcuation?      
Hint: You may use functions in the `scipy.spatial.distance` module on the dense matrix, but it is quite slow (think about the time complexity). If you want to speed up, you may try using functions in the `scipy.sparse` module. 

In [20]:
A = np.array([0,0,1,0,0,0,0,1])
B = np.array([0,1,0,1,0,1,0,1])
C = np.array([0,1,1,0,1,1,0,1])
print(sum([A[i]*B[i] for i in range(len(A))]))
print(sum( [max (A[i],B[i]) for i in range(len(A))]))
jaccard(A,B)
D = csr_matrix([[0,0,1,0,0,0,0,1],[0,1,0,1,0,1,0,1],[0,1,1,0,1,1,0,1]])

1
5


In [21]:
E = cb.Mm
F = np.transpose(E)
G = coo_matrix(E)
# print(G)
H = coo_matrix(F)
# G.dot(H)

In [22]:
cb.calc_item_item_similarity()

c:\Users\eltur\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\metrics\pairwise.py:2466: DataConversionWarning: Data was converted to boolean for metric jaccard
  warnings.warn(msg, DataConversionWarning)


In [23]:
# Sample tests calc_item_item_similarity in ContentBased class 

sample_cb = ContentBased(sample_data)
sample_cb.calc_item_item_similarity() 

# print(np.trace(sample_cb.sim))
# print(sample_cb.sim[10:13,10:13])
assert(sample_cb.sim.sum() > 0), "Check calc_item_item_similarity."
assert(np.trace(sample_cb.sim) == 3152), "Check calc_item_item_similarity. What do you think np.trace(cb.sim) should be?"


ans = np.array([[1, 0.25, 0.],[0.25, 1, 0.],[0., 0., 1]])
for pred, true in zip(sample_cb.sim[10:13, 10:13], ans):
    assert approx(pred, 0.01) == true, "Check calc_item_item_similarity. Look at cb.sim"

c:\Users\eltur\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\metrics\pairwise.py:2466: DataConversionWarning: Data was converted to boolean for metric jaccard
  warnings.warn(msg, DataConversionWarning)


In [24]:
# tests calc_item_item_similarity in ContentBased class 

In [25]:
# additional tests for calc_item_item_similarity in ContentBased class 

In [26]:
# additional tests for calc_item_item_similarity in ContentBased class

In [27]:
# additional tests for calc_item_item_similarity in ContentBased class

In [28]:
# additional tests for calc_item_item_similarity in ContentBased class

### 2c. Complete the function predict_from_sim in the class RecSys [5 pts]

In [29]:
print(sample_cb.Mr.shape)
print(np.sum(sample_cb.Mr[239]))



(5769, 3152)
59


In [30]:
# for a, b in zip(sample_MV_users.uID, sample_MV_movies.mID):
#     print(a, b, sample_cb.predict_from_sim(a,b))

# Sample tests for predict_from_sim in RecSys class 
assert(sample_cb.predict_from_sim(245,276)==approx(2.5128205128205128,abs=1e-2)), "Check predict_from_sim. Look at how you predicted a user rating on a movie given UserID and movieID."
assert(sample_cb.predict_from_sim(2026,2436)==approx(2.785714285714286,abs=1e-2)), "Check predict_from_sim. Look at how you predicted a user rating on a movie given UserID and movieID."

In [31]:
# tests for predict_from_sim in RecSys class 

### 2d. Complete the function predict in the class RecSys [5 pts]
After completing the predict method in the RecSys class, run the cell below to calculate rating prediction and RMSE. How much does the performance increase compared to the baseline results from above? 

In [32]:
# Sample tests method predict in the RecSys class 

sample_yp = sample_cb.predict()
sample_rmse = sample_cb.rmse(sample_yp)
print(sample_rmse)

assert(sample_rmse==approx(1.1962537249116723, abs=1e-2)), "Check method predict in the RecSys class."

1.1962537249116723


In [33]:
# Hidden tests method predict in the RecSys class 

yp = cb.predict()
rmse = cb.rmse(yp)
print(rmse)

1.0128116783754684


In [34]:
# tests method predict in the RecSys class 

# Q3. Collaborative Filtering

### 3a. Complete the function cossim in the class Collaborative [10 pts]
**To Do:**    
1.Impute the unrated entries in self.Mr to the user's average rating then subtract by the user mean, call this matrix X.   
2.Calculate cosine similarity for all item-item pairs. Don't forget to rescale the cosine similarity to be 0~1.    
You might encounter divide by zero warning (numpy will fill nan value for that entry). In that case, you can fill those with appropriate values.    

Hint: Let's say a movie item has not been rated by anyone. When you calculate similarity of this vector to anoter, you will get $\vec{0}$=[0,0,0,....,0]. When you normalize this vector, you'll get divide by zero warning and it will make nan value in self.sim matrix. Theoretically what should the similarity value for $\vec{x}_i \cdot \vec{x}_i$ when $\vec{x}_i = \vec{0}$? What about $\vec{x}_i \cdot \vec{x}_j$ when $\vec{x}_i = \vec{0}$ and $\vec{x}_j$ is an any vector?     

Hint: You may use `scipy.spatial.distance.cosine`, but it will be slow because its cosine function does vector-vector operation whereas you can implement matrix-matrix operation using numpy to calculate all cosines all at once (it can be 100 times faster than vector-vector operation in our data). Also pay attention to the definition. The scipy.spatial.distance provides distance, not similarity. 

3. Run the below cell that calculate yp and RMSE. 

In [35]:
sample_cf = Collaborative(sample_data)
sample_cf.calc_item_item_similarity(sample_cf.cossim)
sample_yp = sample_cf.predict()
sample_rmse = sample_cf.rmse(sample_yp)

print(sample_rmse)

(3152, 3152)
2285.0
3152.0
3.1786134191443445


In [36]:
# Sample tests cossim method in the Collaborative class
##?? This is where I'm stuck.  I also have not looked at the Jaccard similarity question at all.
sample_cf = Collaborative(sample_data)
sample_cf.calc_item_item_similarity(sample_cf.cossim)
sample_yp = sample_cf.predict()
sample_rmse = sample_cf.rmse(sample_yp)
print(sample_rmse)

assert(np.trace(sample_cf.sim)==3152), "Check cossim method in the Collaborative class. What should np.trace(cf.sim) equal?"
assert(sample_rmse==approx(1.1429596846619763, abs=5e-3)), "Check cossim method in the Collaborative class. rmse result is not as expected."
assert(sample_cf.sim[0,:3]==approx([1., 0.5, 0.5],abs=1e-2)), "Check cossim method in the Collaborative class. cf.sim isn't giving the expected results."

(3152, 3152)
2285.0
3152.0
3.1786134191443445


AssertionError: Check cossim method in the Collaborative class. rmse result is not as expected.

In [ ]:
# Hidden tests cossim method in the Collaborative class

cf = Collaborative(data)
cf.calc_item_item_similarity(cf.cossim)
yp = cf.predict()
rmse = cf.rmse(yp)
print(rmse)

NameError: name 'Collaborative' is not defined

In [ ]:
# tests cossim method in the Collaborative class 

In [ ]:
# additional tests for cossim method in the Collaborative class

In [ ]:
# additional tests for cossim method in the Collaborative class

In [ ]:
# additional tests for cossim method in the Collaborative class

In [ ]:
# additional tests for cossim method in the Collaborative class

In [ ]:
# additional tests for cossim method in the Collaborative class

### 3b. Complete the function jacsim in the class Collaborative [15 pts]
**3b [15 pts] = 3b-i) [5 pts]+3b-ii) [5 pts]+ 3b-iii) [5 pts]**

Function `jacsim` calculates jaccard similarity between items using collaborative filtering method. When we have a rating matrix `self.Mr`, the entries of Mr matrix are 0 to 5 (0: unrated, 1-5: rating). We are interested to see which threshold method works better when we use jaccard dimilarity in the collaborative filtering.    
We may treat any rating 3 or above to be 1 and the negatively rated (below 3) and no-rating as 0. Or, we may treat movies with any ratings to be 1 and ones that has no rating as 0. In this question, we will complete a function jacsim that takes a transformed rating matrix X and calculate and returns a jaccard similarity matrix.     
Let's consider these input cases for the utility matrix $M_r$ with ratings 1-5 and 0s for no-rating.    
1. $M_r \geq 3$ 
2. $M_r \geq 0$ 
3. $M_r$, no transform.

Things to think about: 
- The cases 1 and 2 are straightforward to calculate Jaccard, but what does Jaccard mean for multicategory data?
- Time complexity: The matrix $M_r$ is much bigger than the item feature matrix $M_m$, therefore it will take very long time if we calculate on dense matrix.     
Hint: Use sparse matrix.
- Which method will give the best performance?

### 3b-i)  When $M_r\geq3$ [5 pts]
After you've implemented the jacsim function, run the code below. If implemented correctly, you'll have RMSE below 0.99. 

In [ ]:
cf = Collaborative(data)
Xr = cf.Mr>=3
t0=time.perf_counter()
cf.calc_item_item_similarity(cf.jacsim,Xr)
t1=time.perf_counter()
time_sim = t1-t0
print('similarity calculation time',time_sim)
yp = cf.predict()
rmse = cf.rmse(yp)
print(rmse)
assert(rmse<0.99)

In [ ]:
# tests RMSE for jacsim implementation

In [ ]:
# additional tests for RMSE for jacsim implementation

In [ ]:
# additional tests for jacsim implementation

In [ ]:
# additional tests for jacsim implementation

### 3b-ii)  When $M_r\geq1$ [5 pts]
After you've implemented the jacsim function, run the code below. If implemented correctly, you'll have RMSE below 1.0. 

In [ ]:
cf = Collaborative(data)
Xr = cf.Mr>=1
t0=time.perf_counter()
cf.calc_item_item_similarity(cf.jacsim,Xr)
t1=time.perf_counter()
time_sim = t1-t0
print('similarity calculation time',time_sim)
yp = cf.predict()
rmse = cf.rmse(yp)
print(rmse)
assert(rmse<1.0)

In [ ]:
# tests RMSE for jacsim implementation 

In [ ]:
# tests RMSE for jacsim implementation

In [ ]:
# tests jacsim implementation

In [ ]:
# tests performance of jacsim implementation

### 3b-iii)  When $M_r$; no transform [5 pts]
After you've implemented the jacsim function, run the code below. If implemented correctly, you'll have RMSE below 0.96

In [ ]:
cf = Collaborative(data)
Xr = cf.Mr.astype(int)
t0=time.perf_counter()
cf.calc_item_item_similarity(cf.jacsim,Xr)
t1=time.perf_counter()
time_sim = t1-t0
print('similarity calculation time',time_sim)
yp = cf.predict()
rmse = cf.rmse(yp)
print(rmse)
assert(rmse<0.96)

In [ ]:
# tests jacsim implementation RMSE

In [ ]:
# tests jacsim implementation RMSE

In [ ]:
# tests jacsim implementation

In [ ]:
# tests jacsim implementation performance

### 3.C Discussion [Peer Review]
Answer the questions below in this week's Peer Review assignment. <br>
1. Summarize the methods and performances: Below is a template/example.

|Method|RMSE|
|:----|:--------:|
|Baseline, $Y_p$=3| |
|Baseline, $Y_p=\mu_u$| |
|Content based, item-item| |
|Collaborative, cosine| |
|Collaborative, jaccard, $M_r\geq 3$|  |
|Collaborative, jaccard, $M_r\geq 1$|  |
|Collaborative, jaccard, $M_r$|  |

2. Discuss which method(s) work better than others and why.